# 17 · A/B testing e inferencia causal para Data Science

Machine Learning suele responder **¿qué va a pasar?**. Inferencia causal intenta responder **¿qué pasaría si intervengo?**. Una feature importante para predecir no necesariamente es una buena palanca de intervención.

## Objetivos
- Diferenciar correlación, predicción y causalidad.
- Entender potential outcomes y ATE.
- Diseñar un experimento A/B.
- Calcular diferencia de medias, intervalo y test.
- Entender power, sample size, multiple testing y peeking.
- Introducir confounding, propensity scores, matching/IPW y Difference-in-Differences.


## 1. Potential outcomes
Para cada unidad imaginamos $Y(1)$ si recibe tratamiento y $Y(0)$ si no. El efecto individual sería $Y(1)-Y(0)$, pero nunca observamos ambos a la vez. Buscamos cantidades como:
$$ATE=E[Y(1)-Y(0)]$$

La aleatorización hace que, en expectativa, grupos sean comparables y permite estimar causalidad sin modelar todos los confounders.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
SEED=42; rng=np.random.default_rng(SEED); n=6000
treat=rng.binomial(1,.5,n); baseline=rng.normal(50,10,n); true_effect=2.5
y=baseline+true_effect*treat+rng.normal(0,8,n)
df=pd.DataFrame({'treat':treat,'y':y,'baseline':baseline})
df.groupby('treat').y.agg(['count','mean','std'])

## 2. Diferencia de medias e intervalo
En un RCT simple, la diferencia de medias es un estimador insesgado del ATE. El intervalo comunica incertidumbre mejor que un p-value aislado.


In [ ]:
a=df.loc[df.treat==1,'y']; b=df.loc[df.treat==0,'y']; diff=a.mean()-b.mean(); se=np.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b)); ci=(diff-1.96*se,diff+1.96*se); t,p=stats.ttest_ind(a,b,equal_var=False)
print('efecto estimado',diff,'95% CI',ci,'p=',p)

## 3. Significancia estadística ≠ importancia práctica
Con millones de observaciones, efectos minúsculos pueden ser significativos. Antes del experimento define MDE (minimum detectable effect) y métrica primaria. Considera costo, riesgo y efecto absoluto.

Errores:
- Tipo I: falso positivo, controlado por $\alpha$.
- Tipo II: falso negativo, $\beta$; power=$1-\beta$.


In [ ]:
from statsmodels.stats.power import TTestIndPower
power=TTestIndPower();
for effect_size in [.1,.2,.3]:
 n_per=power.solve_power(effect_size=effect_size,alpha=.05,power=.8,ratio=1); print('Cohen d',effect_size,'n/grupo',int(np.ceil(n_per)))

## 4. CUPED / ajuste por covariables
Una variable pre-tratamiento correlacionada con outcome puede reducir variance. Una regresión `Y ~ treatment + baseline` conserva interpretación causal en un experimento randomizado y suele ganar precisión. Nunca ajustes por variables causadas por el tratamiento (post-treatment bias).


In [ ]:
X=sm.add_constant(df[['treat','baseline']]); fit=sm.OLS(df.y,X).fit(cov_type='HC3'); print(fit.summary().tables[1])

## 5. Peeking y multiple testing
Detener apenas `p<0.05` infla falsos positivos. Opciones: tamaño fijo predefinido, sequential testing formal, alpha spending o métodos bayesianos.

Si pruebas 20 métricas/segmentos, la probabilidad de al menos un falso positivo aumenta. Correcciones: Bonferroni/Holm, FDR, o —mejor— predefinir hipótesis primaria.


## 6. Cuando no puedes randomizar: confounding
Si quienes reciben tratamiento son sistemáticamente distintos, comparar medias mezcla efecto del tratamiento con diferencias previas. Métodos observacionales requieren supuestos no verificables completamente.

### Propensity score
$e(x)=P(T=1|X=x)$. Puede usarse para matching, stratification o inverse probability weighting. Requiere **overlap/positivity** y que todos los confounders relevantes estén medidos.


In [ ]:
from sklearn.linear_model import LogisticRegression
# ejemplo confounded: edad aumenta probabilidad de tratamiento y outcome
n=8000; age=rng.normal(45,12,n); p_t=1/(1+np.exp(-(-2+.05*(age-45)))); T=rng.binomial(1,p_t); Y=5+.12*age+2*T+rng.normal(0,5,n)
obs=pd.DataFrame({'age':age,'T':T,'Y':Y}); print('naive',obs[obs.T==1].Y.mean()-obs[obs.T==0].Y.mean())
ps=LogisticRegression().fit(obs[['age']],T).predict_proba(obs[['age']])[:,1]; w=T/ps+(1-T)/(1-ps)
ate=(np.sum(w*T*Y)/np.sum(w*T))-(np.sum(w*(1-T)*Y)/np.sum(w*(1-T))); print('IPW',ate,'true=2')

## 7. Difference-in-Differences (DiD)
Si un grupo recibe una política/intervención en un momento y otro no, DiD compara el cambio antes/después entre grupos. El supuesto clave es **parallel trends** en ausencia del tratamiento. Event studies ayudan a revisar pre-trends.

## 8. Heterogeneous Treatment Effects
El efecto promedio puede ocultar subgrupos. Causal forests, meta-learners (S/T/X/R learners) y uplift models estiman heterogeneidad, pero segmentar post-hoc puede overfit.

## Errores comunes
- interpretar SHAP/feature importance como causal;
- ajustar por variables post-tratamiento;
- ignorar interference entre unidades;
- p-hacking y múltiples segmentos;
- propensity scores sin revisar overlap/balance;
- DiD sin parallel trends;
- reportar solo p-value.

## Ejercicios
1. Simula un A/B con efecto 0 y mide falsos positivos en 1000 experimentos.
2. Implementa Bonferroni y Benjamini-Hochberg.
3. Calcula power para una métrica binaria.
4. Implementa matching por propensity score.
5. Grafica balance de covariables antes/después de IPW.
6. Simula DiD y luego viola parallel trends.
7. Investiga Double/Debiased ML y EconML/CausalML.
